# 8 - Streaming: feature ablation (time / geo)

Never holds the whole cube: the region is cached once to a disk-backed memmap, then grid tiles stream
through the model. This run tests what the model needs to stop filling swaths with a flat average and to
beat persistence on a diverse basin:

- **Feature ablation**: `none` (no time, no geo), `time` (sin/cos day-of-year), `geo` (x/y/z position on the
  unit sphere), `both`.
- **Coast threshold** `MIN_OCEAN`: lowered from 0.5 so valuable coastal tiles are not dropped.
- **Overlap** `OVERLAP`: optional overlapping grid tiles.
- Whole-domain prediction (one forward pass, no tiling seams).

Geo lets a data-free swath fill with the position-and-season average instead of one global number, and the
coastal pixels are where the model can actually beat persistence (open ocean is near-trivial for it).
Random 80/20 train/val split, val only. Geo channels are built inline, so no package push is needed.

## Setup

In [ ]:
%pip install --force-reinstall --no-cache-dir "git+https://github.com/SAFS-Varanasi-Internship/mindthegap.git@troy-branch"
!pip install -qU icechunk

In [ ]:
import os
os.environ.pop("TF_CUDNN_DETERMINISTIC", None)
os.environ.pop("TF_CUDNN_USE_AUTOTUNE", None)
import earthaccess
import icechunk as ic
import numpy as np, pandas as pd
import xarray as xr
import tensorflow as tf
from scipy import ndimage
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs, cartopy.feature as cfeature
import mindthegap as mtg

for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

def create_ds(product="PACE_OCI_L3M_CHL", group="daily/0p1deg/chunks_512"):
    """Open a PACE Icechunk store group as xarray (Eli's helper). Needs AWS us-west-2 + earthaccess login."""
    url = f"https://data.source.coop/fish-pace/pace-oci/inregion/{product}"
    storage = ic.http_storage(url)
    auth = earthaccess.login()
    creds = auth.get_s3_credentials(daac="OBDAAC")
    vc = ic.credentials.containers_credentials({
        "s3://ob-cumulus-prod-public/": ic.credentials.s3_credentials(
            access_key_id=creds["accessKeyId"], secret_access_key=creds["secretAccessKey"],
            session_token=creds["sessionToken"])})
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=vc).readonly_session("main").store
    return xr.open_zarr(store, consolidated=False, group=group, chunks={})

ds = create_ds()
CHL = "chlor_a"
print("global grid:", dict(ds.sizes))

## Region + memmap cache

In [ ]:
LAT_HI, LAT_LO = 31, 5           # Arabian Sea (nb7 region): coastal / dynamic, where the model won before
LON_LO, LON_HI = 42, 80
COMPOSITE_DAYS = 1

reg = ds[CHL].sel(lat=slice(LAT_HI, LAT_LO), lon=slice(LON_LO, LON_HI))
if COMPOSITE_DAYS > 1:
    reg = reg.resample(time=f"{COMPOSITE_DAYS}D").mean()
nlat, nlon = reg.sizes["lat"], reg.sizes["lon"]
reg = reg.isel(lat=slice(0, nlat - nlat % 8), lon=slice(0, nlon - nlon % 8))
times = reg.time.values
T, H, W = reg.sizes["time"], reg.sizes["lat"], reg.sizes["lon"]
LATV, LONV = reg.lat.values, reg.lon.values
print(f"region {H}x{W}, {T} frames  ->  cache is {T*H*W*4/1e9:.2f} GB on DISK")

In [ ]:
CACHE = f"pace_cache/io_{H}x{W}_{T}_c{COMPOSITE_DAYS}.npy"
os.makedirs("pace_cache", exist_ok=True)
if os.path.exists(CACHE):
    mm = np.lib.format.open_memmap(CACHE, mode="r"); print("reusing cache", CACHE, mm.shape)
else:
    mm = np.lib.format.open_memmap(CACHE, mode="w+", dtype="float32", shape=(T, H, W))
    for t0 in range(0, T, 20):
        blk = reg.isel(time=slice(t0, t0 + 20)).values
        mm[t0:t0 + 20] = np.log(np.where(blk > 0, blk, np.nan)).astype("float32")
        print("cached", min(t0 + 20, T), "/", T, flush=True)
    mm.flush(); print("cache built:", CACHE)

ocean = np.zeros((H, W), bool)
for t0 in range(0, T, 50):
    ocean |= np.isfinite(mm[t0:t0 + 50]).any(0)
ext = [float(reg.lon.min()), float(reg.lon.max()), float(reg.lat.min()), float(reg.lat.max())]
print("ocean:", f"{ocean.mean():.0%}")

## Split, tiles, stats

Random 80/20 train/val. `MIN_OCEAN` sets how much coast survives (0.5 dropped a lot of coastline; lower it
to keep the dynamic coastal pixels). `OVERLAP` optionally overlaps grid tiles. Stats are computed once on
the base channels (geo is appended per config, so it is not standardized).

In [ ]:
COVERAGE = 0.2; CS = 4; N_DAYS = 3
BLOB_SIGMA = 12          # fake-cloud SIZE (default was 6): bigger = larger clouds, SAME 20% coverage. Try 18.
CHUNK_H, CHUNK_W = 40, 56
MIN_OCEAN = 0.2          # coast threshold: LOWER than 0.5 keeps more coastal (dynamic) tiles
OVERLAP = 0             # grid tile overlap in px (0 = non-overlap); try 20 for overlapping tiles
BATCH = 16; STEPS_PER_EPOCH = 200

rng0 = np.random.default_rng(0)
perm = rng0.permutation(T); ntr = int(0.8 * T)
tr = np.zeros(T, bool); tr[perm[:ntr]] = True; va = ~tr
train_frames = np.where(tr)[0]; val_frames = np.where(va)[0]
print("random split -> train", int(tr.sum()), "val", int(va.sum()), "(no test)")

step_y, step_x = CHUNK_H - OVERLAP, CHUNK_W - OVERLAP
grid_pos = [(yy, xx) for yy in range(0, H - CHUNK_H + 1, step_y)
                     for xx in range(0, W - CHUNK_W + 1, step_x)
                     if ocean[yy:yy+CHUNK_H, xx:xx+CHUNK_W].mean() >= MIN_OCEAN]
print(len(grid_pos), f"tiles (MIN_OCEAN={MIN_OCEAN}, OVERLAP={OVERLAP})")

coarse = np.asarray(mm[:, ::CS, ::CS])
_, _, STATS, ORDER = mtg.build_pace_channels(coarse, times, tr, n_days=N_DAYS, cloud_mode="synthetic",
                                             coverage=COVERAGE, blob_sigma=BLOB_SIGMA, time_sigma=1.5, seed=0)   # base channels (no geo)
Y_MEAN, Y_STD = STATS["CHL"]; del coarse
print("base channels:", len(ORDER), "\n", ORDER)

## Grid-tile check (colorblind-safe)

Filled dot = tile that trains, x = dropped (below `MIN_OCEAN` or in the edge strip). Lowering `MIN_OCEAN`
should light up more coastline as trained tiles.

In [ ]:
all_pos = [(yy, xx) for yy in range(0, H - CHUNK_H + 1, step_y)
                    for xx in range(0, W - CHUNK_W + 1, step_x)]
kept = set(grid_pos)
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(np.where(ocean, 0.9, 0.35), cmap="gray", vmin=0, vmax=1, origin="upper")
for (yy, xx) in all_pos:
    cy, cx = yy + CHUNK_H/2, xx + CHUNK_W/2
    if (yy, xx) in kept:
        ax.add_patch(mpatches.Rectangle((xx, yy), CHUNK_W, CHUNK_H, fill=False, edgecolor="black", lw=1.0))
        ax.plot(cx, cy, "o", ms=5, color="black")
    else:
        ax.add_patch(mpatches.Rectangle((xx, yy), CHUNK_W, CHUNK_H, fill=False, edgecolor="black", lw=0.5, ls=":"))
        ax.plot(cx, cy, "x", ms=7, color="black", mew=1.6)
ax.set_title(f"dot = trained ({len(kept)}), x = dropped ({len(all_pos)-len(kept)})  |  MIN_OCEAN={MIN_OCEAN}, OVERLAP={OVERLAP}")
ax.set_xlabel("lon index"); ax.set_ylabel("lat index"); plt.tight_layout(); plt.show()

covered = np.zeros((H, W), bool)
for (yy, xx) in grid_pos: covered[yy:yy+CHUNK_H, xx:xx+CHUNK_W] = True
oc = int(ocean.sum())
print(f"ocean covered by tiles: {(covered & ocean).sum()/oc:5.1%}   not covered: {((~covered) & ocean).sum()/oc:5.1%}")

## Feature configs

`build_pace_channels` builds the base channels (masked CHL, prev/next, flags) plus sin/cos time. Each config
selects which of those to feed and whether to append geo (x/y/z position), so all four share one channel
build.

In [ ]:
CONFIGS = ["none", "time", "geo", "both"]
def keep_base(cfg):
    drop = set()
    if cfg in ("none", "geo"):  drop |= {"sin_time", "cos_time"}     # no time
    return [c for c in ORDER if c not in drop]
def use_geo(cfg): return cfg in ("geo", "both")
def nc_of(cfg):   return len(keep_base(cfg)) + (3 if use_geo(cfg) else 0)

def geo_of(yy, xx, th, tw):
    latr = np.deg2rad(LATV[yy:yy+th])[:, None]; lonr = np.deg2rad(LONV[xx:xx+tw])[None, :]
    px = np.broadcast_to(np.cos(latr) * np.cos(lonr), (th, tw))
    py = np.broadcast_to(np.cos(latr) * np.sin(lonr), (th, tw))
    pz = np.broadcast_to(np.sin(latr), (th, tw))
    return np.stack([px, py, pz], -1).astype("float32")             # (th, tw, 3) in [-1, 1]

def build_XY(yy, xx, seed, cfg):
    cc = np.asarray(mm[:, yy:yy+CHUNK_H, xx:xx+CHUNK_W])
    ch, y, _, _ = mtg.build_pace_channels(cc, times, tr, n_days=N_DAYS, cloud_mode="synthetic",
                                          coverage=COVERAGE, blob_sigma=BLOB_SIGMA, time_sigma=1.5, seed=seed, stats=STATS)
    Xb = np.stack([ch[k] for k in keep_base(cfg)], -1).astype("float32")   # (T, th, tw, nbase)
    if use_geo(cfg):
        g = geo_of(yy, xx, CHUNK_H, CHUNK_W)
        Xb = np.concatenate([Xb, np.broadcast_to(g[None], Xb.shape[:-1] + (3,))], -1).astype("float32")
    return Xb, mtg.target_with_mask(y)

for cfg in CONFIGS:
    print(f"{cfg:>5}: {nc_of(cfg)} channels")

## Train the four configs (resumable)

Each model saves and is skipped if present. If a later config OOMs (four trainings back to back), restart
the kernel and re-run: it loads the finished ones.

In [ ]:
def make_gen(cfg):
    def gen():
        rng = np.random.default_rng(1)
        while True:
            tiles = list(grid_pos); rng.shuffle(tiles)
            for (yy, xx) in tiles:
                X, Y = build_XY(yy, xx, int(rng.integers(10**9)), cfg)
                fo = train_frames.copy(); rng.shuffle(fo)
                for d in fo:
                    yield X[d], Y[d]
    return gen

def train_config(cfg):
    name = f"feat_{cfg}"; path = f"models/pace/io_{H}x{W}_{T}_feat_{cfg}.keras"; NC = nc_of(cfg)   # region-keyed
    if os.path.exists(path):
        print("load", name, flush=True); return name, tf.keras.models.load_model(path, compile=False), cfg
    Xvl, Yvl = [], []
    for (yy, xx) in grid_pos[::max(1, len(grid_pos)//8)][:8]:
        X, Y = build_XY(yy, xx, 7, cfg); Xvl.append(X[val_frames]); Yvl.append(Y[val_frames])
    Xva = np.concatenate(Xvl).astype("float32"); Yva = np.concatenate(Yvl).astype("float32")
    sig = (tf.TensorSpec((CHUNK_H, CHUNK_W, NC), tf.float32), tf.TensorSpec((CHUNK_H, CHUNK_W, 2), tf.float32))
    ds_tr = (tf.data.Dataset.from_generator(make_gen(cfg), output_signature=sig)
             .shuffle(2048).batch(BATCH, drop_remainder=True).prefetch(2))
    model = mtg.UNet((None, None, NC)); model.compile("adam", loss=mtg.masked_mse, jit_compile=False)
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
    print(f"\n===== training {name} (NC={NC}) =====", flush=True)
    model.fit(ds_tr, steps_per_epoch=STEPS_PER_EPOCH, validation_data=(Xva, Yva),
              epochs=40, callbacks=[es], verbose=2)
    os.makedirs("models/pace", exist_ok=True); model.save(path); del Xva, Yva
    return name, model, cfg

MODELS = {}
for cfg in CONFIGS:
    nm, model, c = train_config(cfg); MODELS[nm] = (model, c)
print("\nready:", list(MODELS))

## Compare: held-out MAE vs persistence

Whole-domain prediction (one pass, no tiling seams). Each config scored on the same val frames vs standard
persistence. A coastal-vs-open split shows where each config helps, since open ocean is near-trivial.

In [ ]:
PRED_TH = max(8, (H // 3 // 8) * 8); PRED_TW = max(8, (W // 3 // 8) * 8)   # only the OOM fallback
def _predict(model, X):
    try:
        return model.predict(X[None], verbose=0)[0, ..., 0] * Y_STD + Y_MEAN      # whole domain, ONE pass
    except tf.errors.ResourceExhaustedError:
        Hh, Ww, _ = X.shape; th = (min(Hh, PRED_TH)//8)*8; tw = (min(Ww, PRED_TW)//8)*8
        sh, sw = th//2, tw//2; wgt = np.outer(np.hanning(th), np.hanning(tw)) + 1e-3
        acc = np.zeros((Hh, Ww), np.float32); cnt = np.zeros((Hh, Ww), np.float32)
        yl = list(range(0, Hh-th+1, sh)) or [0]; xl = list(range(0, Ww-tw+1, sw)) or [0]
        if yl[-1] != Hh-th: yl.append(Hh-th)
        if xl[-1] != Ww-tw: xl.append(Ww-tw)
        for a in yl:
            for b in xl:
                p = model.predict(X[a:a+th, b:b+tw][None], verbose=0)[0, :, :, 0]
                acc[a:a+th, b:b+tw] += p*wgt; cnt[a:a+th, b:b+tw] += wgt
        return acc/np.maximum(cnt, 1e-6) * Y_STD + Y_MEAN

geo_full = geo_of(0, 0, H, W)
def predict_day(model, cfg, d):
    lo = max(0, d-N_DAYS); hi = min(T, d+N_DAYS+1); mid = int(d-lo)
    win = np.asarray(mm[lo:hi])
    ch, y, _, _ = mtg.build_pace_channels(win, times[lo:hi], np.ones(win.shape[0], bool),
                                          n_days=N_DAYS, coverage=COVERAGE, blob_sigma=BLOB_SIGMA, time_sigma=1.5,
                                          seed=1000+int(d), stats=STATS, land=~ocean)
    faked = ch["fake_cloud_flag"][mid].astype(bool); truth = win[mid]
    Xd = np.stack([ch[k] for k in keep_base(cfg)], -1)[mid]
    if use_geo(cfg): Xd = np.concatenate([Xd, geo_full], -1)
    return truth, faked, _predict(model, Xd.astype("float32"))

# coastal vs open masks
coast = ocean & (ndimage.distance_transform_edt(ocean) <= 8)
openoc = ocean & ~coast
SCORE = np.where(va)[0]; SCORE = SCORE[(SCORE >= 3) & (SCORE < T - 3)][:16]

def zone_mae(pred, truth, faked, zone):
    m = faked & zone & np.isfinite(truth) & np.isfinite(pred)
    return float(np.mean(np.abs(pred[m] - truth[m]))) if m.any() else np.nan

results = {}
for nm, (model, cfg) in MODELS.items():
    rows = {"coast": [], "open": [], "all": [], "pers": []}
    for d in SCORE:
        truth, faked, filled = predict_day(model, cfg, d)
        prev = np.asarray(mm[d - 1])
        rows["coast"].append(zone_mae(filled, truth, faked, coast))
        rows["open"].append(zone_mae(filled, truth, faked, openoc))
        rows["all"].append(zone_mae(filled, truth, faked, ocean))
        rows["pers"].append(zone_mae(prev, truth, faked, ocean))
    results[nm] = {k: float(np.nanmean(v)) for k, v in rows.items()}
persist = float(np.nanmean([results[n]["pers"] for n in results]))

print(f"{'config':>10} {'coast':>8} {'open':>8} {'all':>8}")
for nm, r in results.items():
    print(f"{nm:>10} {r['coast']:>8.4f} {r['open']:>8.4f} {r['all']:>8.4f}")
print(f"{'persist':>10} {'':8} {'':8} {persist:>8.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
names = list(results); vals = [results[n]["all"] for n in names]
ax.bar(names, vals, color="#0F9E86")
for i, v in enumerate(vals): ax.text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=9)
ax.axhline(persist, ls="--", color="#C4623B", lw=1.5, label=f"persistence {persist:.3f}")
ax.set_ylabel("held-out MAE, log Chl-a  (lower is better)"); ax.legend(frameon=False)
ax.set_title(f"feature ablation  (MIN_OCEAN={MIN_OCEAN}, OVERLAP={OVERLAP})")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("compare_features.png", dpi=140, bbox_inches="tight"); plt.show()

## 12 days across the year, one grid per config

Same 12 val days for every config, shared color scale. Watch the swaths: `geo` and `both` should fill them
with a smoother, position-appropriate field instead of one flat number.

In [ ]:
months = pd.to_datetime(times).month; idx = np.arange(T)
elig = va & (idx >= 3) & (idx < T - 3)
chosen = []
for m in range(1, 13):
    cand = np.where(elig & (months == m))[0]
    if len(cand) == 0: cand = np.where((months == m) & (idx >= 3) & (idx < T - 3))[0]
    if len(cand) == 0: continue
    obs = np.array([np.isfinite(mm[d])[ocean].mean() for d in cand])
    chosen.append(int(cand[np.argmin(np.abs(obs - 0.5))]))
print("days:", [str(pd.to_datetime(times[d]).date()) for d in chosen])

store = {nm: {d: predict_day(MODELS[nm][0], MODELS[nm][1], d) for d in chosen} for nm in MODELS}
vmin, vmax = np.nanpercentile(np.concatenate([np.asarray(mm[d])[ocean] for d in chosen]), [2, 98])
allerr = np.concatenate([np.abs(store[n][d][2] - store[n][d][0])[store[n][d][1]] for n in MODELS for d in chosen])
emax = float(np.nanpercentile(allerr[np.isfinite(allerr)], 95))

def plot_year(nm):
    tt = ["input (clouds hidden)", "U-Net filled", "truth", "abs error @ fake"]
    fig, axes = plt.subplots(len(chosen), 4, figsize=(15, 2.6 * len(chosen)))
    for i, d in enumerate(chosen):
        truth, faked, filled = store[nm][d]; holes = faked | ~np.isfinite(truth)
        arrs = [np.where(holes, np.nan, truth), np.where(ocean, filled, np.nan),
                np.where(ocean, truth, np.nan), np.where(faked, np.abs(filled - truth), np.nan)]
        for j, arr in enumerate(arrs):
            ax = axes[i, j]; cm, l, h = ("magma", 0, emax) if j == 3 else ("viridis", vmin, vmax)
            ax.imshow(arr, cmap=cm, vmin=l, vmax=h, extent=ext, origin="upper"); ax.axis("off")
            if i == 0: ax.set_title(tt[j], size=11)
        axes[i, 0].text(-0.04, 0.5, str(pd.to_datetime(times[d]).date()), transform=axes[i, 0].transAxes,
                        rotation=90, va="center", ha="right", size=9)
    fig.subplots_adjust(right=0.9, hspace=0.05, wspace=0.03)
    c1 = fig.add_axes([0.92, 0.35, 0.011, 0.4]); fig.colorbar(
        plt.cm.ScalarMappable(plt.Normalize(vmin, vmax), "viridis"), cax=c1, label="log Chl-a")
    c2 = fig.add_axes([0.965, 0.35, 0.011, 0.4]); fig.colorbar(
        plt.cm.ScalarMappable(plt.Normalize(0, emax), "magma"), cax=c2, label="abs error")
    plt.suptitle(f"{nm}: gap-fill across the year, {H}x{W}", y=0.9, size=13)
    plt.savefig(f"gapfill_year_{nm}.png", dpi=120, bbox_inches="tight"); plt.show()

for nm in MODELS:
    plot_year(nm)

## Next steps

- If geo wins, promote the `coords` channels into `build_pace_channels` (package) instead of the inline build.
- Sweep `MIN_OCEAN` and `OVERLAP` once the feature question is settled.
- Widen toward the full basin; cos-latitude tile weighting at high latitudes.